# Support Vector Regression (SVR)

**Repositorio:** `autodidact/both/svm/`

---

## 1. Idea General e Intuición

**SVR** (Support Vector Regression) adapta la idea de SVM al problema de regresión. En lugar de buscar un hiperplano que *separe* dos clases, busca una **función** que se ajuste a los datos **dentro de un tubo de tolerancia $\varepsilon$**.

**Intuición:** SVR construye un "tubo" de ancho $2\varepsilon$ alrededor de la función predicha. Los puntos *dentro* del tubo no tienen error. Solo los puntos *fuera* del tubo penalizan al modelo. El objetivo es hacer el tubo lo más ancho posible (similar al margen en clasificación) mientras minimiza los errores de los puntos fuera.

## 2. Fundamento Matemático

### 2.1 Función de Pérdida $\varepsilon$-Insensible

$$L_\varepsilon(y, \hat{y}) = \max(0, |y - \hat{y}| - \varepsilon)$$

Solo hay pérdida si el error absoluto supera $\varepsilon$. Dentro del tubo la pérdida es **cero**.

### 2.2 Problema de Optimización

Con variables de holgura $\xi_i, \xi_i^* \geq 0$ para puntos fuera del tubo:

$$\underset{\mathbf{w}, b, \xi, \xi^*}{\min} \quad \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i=1}^{n}(\xi_i + \xi_i^*)$$

Sujeto a:
$$y_i - \mathbf{w}^\top \mathbf{x}_i - b \leq \varepsilon + \xi_i$$
$$\mathbf{w}^\top \mathbf{x}_i + b - y_i \leq \varepsilon + \xi_i^*$$
$$\xi_i, \xi_i^* \geq 0$$

### 2.3 Predicción

$$\hat{y}(\mathbf{x}) = \sum_{i \in SV} (\alpha_i - \alpha_i^*) K(\mathbf{x}_i, \mathbf{x}) + b$$

Solo los **vectores de soporte** (puntos en el borde o fuera del tubo, con $\alpha_i \neq 0$ o $\alpha_i^* \neq 0$) contribuyen a la predicción.

### 2.4 Parámetros Clave

| Parámetro | Rol | Efecto |
|---|---|---|
| **C** | Regularización (igual que SVC) | ↑C → menos regularización, más ajuste a outliers |
| **ε (epsilon)** | Ancho del tubo de tolerancia | ↑ε → más muestras ignoradas, función más suave |
| **kernel** | Tipo de función kernel | Mismos que SVC: `rbf`, `linear`, `poly` |
| **γ (gamma)** | Parámetro del kernel RBF | ↑γ → función más local, más sobreajuste |

### 2.5 Métricas de Evaluación

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **MSE** | $\frac{1}{n}\sum(y_i-\hat{y}_i)^2$ | Sensible a outliers |
| **MAE** | $\frac{1}{n}\sum\|y_i-\hat{y}_i\|$ | Robusto a outliers |
| **$R^2$** | $1-\frac{SS_{res}}{SS_{tot}}$ | Proporción de varianza explicada |
| **RMSE** | $\sqrt{MSE}$ | Mismas unidades que $y$ |

## 3. ¿Cuándo Usarlo y Cuándo NO?

### ✅ Úsalo cuando:
- Dataset **pequeño o mediano** con relaciones no lineales.
- El target tiene **algunos outliers** que quieres ignorar (tubo ε).
- Necesitas **robustez** en la regresión.
- Las features tienen **interacciones complejas** (kernel RBF).

### ❌ NO lo uses cuando:
- Dataset **muy grande** (SVR es O(n²)~O(n³)).
- Necesitas **extrapolación** (SVR no extrapola bien).
- El target tiene muchos **outliers extremos** (usar HuberRegressor).
- Necesitas **interpretabilidad directa** de coeficientes.

## 5. Problemas Específicos

### 5.1 Escalado del Target
El parámetro ε es absoluto. Si el target tiene escala grande (ej. precios de casas × 100k), un ε=0.1 puede ser irrelevante o demasiado estricto.
**Solución:** Escalar también el target con `StandardScaler` o ajustar ε al rango del target.

### 5.2 Elección de ε
Un ε muy grande → todos los puntos dentro del tubo → modelo trivial (predice la media).
Un ε muy pequeño → todos los puntos son SVs → sobreajuste.
**Regla empírica:** $\varepsilon \approx \sigma_{ruido}$ (desviación estándar del ruido esperado).

### 5.3 Número de Vectores de Soporte
Si la mayoría de los puntos son SVs, el modelo está sobreajustando o ε es muy pequeño.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR, LinearSVR
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
SEED = 42
print('✅ Librerías cargadas.')

---
## 6. Demostración Visual del Tubo ε (1D)

In [ ]:
# =============================================================================
# DATOS SINTÉTICOS 1D: mostrar visualmente el tubo epsilon
# =============================================================================
rng = np.random.RandomState(SEED)
X_1d = np.sort(rng.uniform(0, 5, 100)).reshape(-1, 1)
y_1d = np.sin(X_1d.ravel() * 2) + rng.randn(100) * 0.3

x_plot = np.linspace(0, 5, 500).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (epsilon, C) in zip(axes, [(0.05, 1), (0.3, 1), (0.3, 100)]):
    scaler_1d = StandardScaler()
    X_sc_1d   = scaler_1d.fit_transform(X_1d)
    x_sc_plot = scaler_1d.transform(x_plot)

    svr = SVR(kernel='rbf', C=C, epsilon=epsilon, gamma='scale')
    svr.fit(X_sc_1d, y_1d)
    y_hat = svr.predict(x_sc_plot)
    sv_mask = np.zeros(len(X_1d), dtype=bool)
    sv_mask[svr.support_] = True

    ax.scatter(X_1d[~sv_mask], y_1d[~sv_mask], color='steelblue',
               s=20, alpha=0.6, label='Dentro del tubo')
    ax.scatter(X_1d[sv_mask],  y_1d[sv_mask],  color='crimson',
               s=50, alpha=0.9, label=f'SVs ({sv_mask.sum()})')
    ax.plot(x_plot, y_hat, 'k-', lw=2, label='Predicción')
    ax.fill_between(x_plot.ravel(), y_hat - epsilon, y_hat + epsilon,
                    alpha=0.25, color='gold', label=f'Tubo ε={epsilon}')
    r2 = r2_score(y_1d, svr.predict(X_sc_1d))
    ax.set_title(f'C={C}, ε={epsilon}\nR²={r2:.3f} | SVs={sv_mask.sum()}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8); ax.set_xlabel('x'); ax.set_ylabel('y')

plt.suptitle('SVR: El Tubo ε y los Vectores de Soporte', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Regresión Real — Diabetes Dataset

In [ ]:
# =============================================================================
# CARGA Y SPLIT — Diabetes (pequeño, ideal para SVR)
# =============================================================================
diabetes = load_diabetes(as_frame=True)
X, y = diabetes.data, diabetes.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Target — Media: {y.mean():.1f} | Std: {y.std():.1f}')

In [ ]:
# =============================================================================
# PIPELINE: StandardScaler + SVR
# CRÍTICO: escalar tanto features como target para que ε tenga sentido
# =============================================================================
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

# TransformedTargetRegressor aplica el scaler también al target (y)
svr_pipe = Pipeline([
    ('scaler_X', StandardScaler()),
    ('svr',      SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale'))
])

# Scaler del target por separado para correcta inversión
scaler_y = StandardScaler()
y_train_sc = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_sc  = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

svr_pipe.fit(X_train, y_train_sc)
y_pred_sc = svr_pipe.predict(X_test)

# Invertir el escalado del target para métricas interpretables
y_pred = scaler_y.inverse_transform(y_pred_sc.reshape(-1, 1)).ravel()

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
n_sv = svr_pipe.named_steps['svr'].n_support_[0]

print(f'R²   : {r2:.4f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAE  : {mae:.2f}')
print(f'SVs  : {n_sv} / {len(X_train)} ({100*n_sv/len(X_train):.1f}% del training set)')

In [ ]:
# =============================================================================
# GRID SEARCH: C, epsilon, gamma en malla logarítmica
# =============================================================================
param_grid = {
    'svr__C':       [0.1, 1, 10, 100],
    'svr__epsilon': [0.01, 0.1, 0.5, 1.0],
    'svr__gamma':   ['scale', 0.01, 0.1]
}

pipe_gs = Pipeline([
    ('scaler_X', StandardScaler()),
    ('svr', SVR(kernel='rbf'))
])

grid = GridSearchCV(pipe_gs, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train_sc)

print(f'Mejores parámetros : {grid.best_params_}')
print(f'Mejor R² (CV)      : {grid.best_score_:.4f}')

best_svr = grid.best_estimator_
y_pred_gs_sc = best_svr.predict(X_test)
y_pred_gs    = scaler_y.inverse_transform(y_pred_gs_sc.reshape(-1,1)).ravel()
print(f'Test R²            : {r2_score(y_test, y_pred_gs):.4f}')
print(f'Test RMSE          : {np.sqrt(mean_squared_error(y_test, y_pred_gs)):.2f}')

In [ ]:
# =============================================================================
# VISUALIZACIÓN: Predichos vs Reales + Residuos + Efecto de epsilon
# =============================================================================
residuals = y_test.values - y_pred_gs

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predichos vs reales
axes[0].scatter(y_test, y_pred_gs, alpha=0.5, color='steelblue', s=30)
lim = [y_test.min(), y_test.max()]
axes[0].plot(lim, lim, 'r--', lw=2, label='Predicción perfecta')
axes[0].set_xlabel('Valores Reales'); axes[0].set_ylabel('Predicciones')
axes[0].set_title(f'Predichos vs Reales  ($R^2$={r2_score(y_test, y_pred_gs):.4f})',
                  fontsize=12, fontweight='bold')
axes[0].legend()

# Residuos
axes[1].hist(residuals, bins=30, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='crimson', lw=2, linestyle='--', label='Error=0')
axes[1].set_xlabel('Residuo'); axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Residuos', fontsize=12, fontweight='bold')
axes[1].legend()

plt.suptitle('SVR — Diabetes Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# COMPARATIVA: SVR vs LinearSVR vs Ridge (referencia lineal)
# =============================================================================
from sklearn.linear_model import Ridge

models = {
    'SVR (RBF)':   Pipeline([('sc', StandardScaler()), ('m', SVR(kernel='rbf', C=10, gamma='scale'))]),
    'SVR (Linear)':Pipeline([('sc', StandardScaler()), ('m', SVR(kernel='linear', C=1.0))]),
    'LinearSVR':   Pipeline([('sc', StandardScaler()), ('m', LinearSVR(C=1.0, max_iter=5000))]),
    'Ridge':       Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0))])
}

print(f'{"Modelo":<18}  R² (CV)          RMSE (CV)')
print('-' * 50)
for name, model in models.items():
    r2_cv   = cross_val_score(model, X, y, cv=5, scoring='r2')
    rmse_cv = np.sqrt(-cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error'))
    print(f'{name:<18}  {r2_cv.mean():.4f} ± {r2_cv.std():.4f}  '
          f'{rmse_cv.mean():.2f} ± {rmse_cv.std():.2f}')

---
## 8. Resumen Comparativo SVC vs SVR

| Aspecto | SVC (Clasificador) | SVR (Regresor) |
|---|---|---|
| **Objetivo** | Máximo margen de separación | Tubo ε de mínima penalización |
| **Parámetro especial** | — | **ε** (ancho del tubo) |
| **Salida** | Clase (o probabilidad) | Valor continuo |
| **Vectores de soporte** | Puntos en el margen | Puntos fuera del tubo |
| **Métrica** | Accuracy / ROC-AUC | R² / RMSE / MAE |

### ✅ Checklist SVR:
- [ ] ¿Escalé **tanto features como target**?
- [ ] ¿Ajusté ε al rango de ruido esperado del target?
- [ ] ¿Hice GridSearch en malla log de C, ε y γ?
- [ ] ¿Verifiqué el % de SVs (< 30% es saludable)?
- [ ] ¿Comparé con Ridge/Lasso como baseline lineal?

---
*Ver notebook de clasificación: `svm_clasificador.ipynb`*